<a href="https://colab.research.google.com/github/Karuneshtiwari/ML-lab/blob/main/Lab08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Libraries

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Constants

BASE_PATH = "/content/drive/MyDrive/ML_lab"
CSV_PATH = os.path.join(BASE_PATH, "BreaKHis_metadata.csv")
IMAGE_ROOT = os.path.join(BASE_PATH, "BreaKHis_v1")

IMAGE_SIZE = (16, 16)
RANDOM_STATE = 42
TEST_SIZE = 0.20

LEARNING_RATE = 0.05
MAX_EPOCHS = 1000
CONVERGENCE_ERROR = 0.002

In [2]:
# Common Setup

drive.mount('/content/drive')

metadata = pd.read_csv(CSV_PATH)

metadata["label"] = metadata["filename"].apply(
    lambda path: (
        "benign" if "/benign/" in str(path).lower()
        else "malignant" if "/malignant/" in str(path).lower()
        else None
    )
)

metadata = metadata.dropna(subset=["label"]).reset_index(drop=True)

Mounted at /content/drive


In [3]:
# Common Functions

def summation_unit(inputs, weights):
    inputs = np.asarray(inputs, dtype=float)
    weights = np.asarray(weights, dtype=float)

    return weights[0] + np.dot(weights[1:], inputs)


def step_activation(value):
    return 1 if value >= 0 else 0


def bipolar_step_activation(value):
    if value > 0:
        return 1
    if value < 0:
        return -1
    return 0


def sigmoid_activation(value):
    value = np.clip(value, -500, 500)
    return 1 / (1 + np.exp(-value))


def tanh_activation(value):
    return np.tanh(value)


def relu_activation(value):
    return max(0, value)


def leaky_relu_activation(value, alpha=0.01):
    return value if value >= 0 else alpha * value


def activation_function(value, activation="step"):
    activation = activation.lower()

    activation_functions = {
        "step": step_activation,
        "bipolar": bipolar_step_activation,
        "sigmoid": sigmoid_activation,
        "tanh": tanh_activation,
        "relu": relu_activation,
        "leaky_relu": leaky_relu_activation
    }

    if activation not in activation_functions:
        raise ValueError("Unsupported activation function")

    return activation_functions[activation](value)


def calculate_error(target, output):
    return np.asarray(target, dtype=float) - np.asarray(output, dtype=float)


def calculate_sse(targets, outputs):
    errors = calculate_error(targets, outputs)
    return float(np.sum(errors ** 2))


def locate_image(filename):
    relative_path = str(filename).replace("\\", "/").lstrip("/")

    candidates = [
        os.path.join(BASE_PATH, relative_path),
        os.path.join(IMAGE_ROOT, relative_path)
    ]

    if relative_path.startswith("BreaKHis_v1/"):
        short_path = relative_path.split("BreaKHis_v1/", 1)[1]

        candidates.extend([
            os.path.join(IMAGE_ROOT, short_path),
            os.path.join(IMAGE_ROOT, "BreaKHis_v1", short_path)
        ])

    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate

    return None


def load_project_dataset(metadata):
    features = []
    labels = []
    missing_files = 0

    for _, row in metadata.iterrows():
        image_path = locate_image(row["filename"])

        if image_path is None:
            missing_files += 1
            continue

        try:
            with Image.open(image_path) as image:
                image = image.convert("L")
                image = image.resize(IMAGE_SIZE)

                image_array = np.asarray(
                    image,
                    dtype=np.float32
                ) / 255.0

            features.append(image_array.flatten())
            labels.append(1 if row["label"] == "malignant" else 0)

        except (OSError, ValueError):
            missing_files += 1

    return (
        np.asarray(features, dtype=np.float32),
        np.asarray(labels, dtype=np.int64),
        missing_files
    )

In [5]:
SAMPLES_PER_CLASS = 200

project_subset = (
    metadata.groupby("label", group_keys=False)
    .sample(n=SAMPLES_PER_CLASS, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

X_project, y_project, missing_files = load_project_dataset(project_subset)

print("Loaded Dataset Shape:", X_project.shape)

print("\nClass Distribution:")
print(
    pd.Series(y_project)
    .value_counts()
    .rename(index={0: "benign", 1: "malignant"})
)

print("\nImages not loaded:", missing_files)

if len(X_project) == 0:
    raise ValueError("No images were loaded. Check the dataset paths.")

Loaded Dataset Shape: (400, 256)

Class Distribution:
benign       200
malignant    200
Name: count, dtype: int64

Images not loaded: 0
